# Silver PSGC Population
The Silver layer transforms the nested PSGC population payload from `public_health.bronze.geography_bronze` into clean, typed municipal population observations.

Silver grain: one row per psgc_code × population_year × source_version.


1. Parse the nested population JSON

    The PSGC population observations are stored inside the Bronze `populations_json` field. The parent PSGC code from the Bronze geography record is used as the Silver `psgc_code`.

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW psgc_population_observations AS

SELECT
    g.psgc_code AS psgc_code,
    g.source_version,
    g.content_checksum,
    g.batch_id,
    population_item.year AS population_year,
    population_item.population AS population_raw,

    -- text inside a leading bracket, e.g. 'excluding CITY OF ANGELES'
    NULLIF(
        REGEXP_EXTRACT(TRIM(population_item.population), '^[(]([^)]*)[)]', 1),
        ''
    ) AS population_note,

    -- the value with the bracketed note, commas and spaces removed
    REPLACE(
        REPLACE(
            REGEXP_REPLACE(TRIM(population_item.population), '^[(][^)]*[)]', ''),
            ',', ''
        ),
        ' ', ''
    ) AS population_number_text

FROM public_health.bronze.geography_bronze g

LATERAL VIEW EXPLODE(
    FROM_JSON(
        g.populations_json,
        'array<struct<id:bigint,population:string,psgc:string,year:int>>'
    )
) exploded AS population_item

WHERE g.populations_json IS NOT NULL;

2. Classify population parsing results

    The raw population value is trimmed and converted to a numeric BIGINT.
    - `MISSING`: no population value was supplied
    - `INVALID`: a non-null population value cannot be parsed as a number
    - `VALID`: population was successfully parsed

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW psgc_population_parsed AS

SELECT
    psgc_code,
    population_year,
    CASE WHEN population_number_text RLIKE '^[0-9]+$'
         THEN CAST(population_number_text AS BIGINT) END AS population,
    population_raw,
    population_note,
    CASE
        WHEN population_number_text RLIKE '^[0-9]+$' AND population_note IS NOT NULL THEN 'VALID_WITH_NOTE'
        WHEN population_number_text RLIKE '^[0-9]+$' THEN 'VALID'
        WHEN population_raw IS NULL OR TRIM(population_raw) IN ('', '-') THEN 'MISSING'
        ELSE 'INVALID'
    END AS population_parse_status,
    source_version,
    content_checksum,
    batch_id

FROM psgc_population_observations;

3. Profile the population observations

    This is an inspection step. It does not modify the data.

    The Bronze QC identified 94 non-null population values that could not be parsed as BIGINT; these records are expected to be excluded from the clean Silver population table.

In [0]:
%sql

SELECT
    source_version,
    COUNT(*) AS total_observations,
    COUNT_IF(population_parse_status = 'VALID') AS valid_population,
    COUNT_IF(population_parse_status = 'MISSING') AS missing_population,
    COUNT_IF(population_parse_status = 'INVALID') AS invalid_population,
    COUNT_IF(population_parse_status = 'VALID_WITH_NOTE') AS valid_with_note,
    COUNT(DISTINCT psgc_code) AS distinct_psgc_codes
FROM psgc_population_parsed
GROUP BY source_version
ORDER BY source_version;

4. Check the Silver grain

    The intended Silver grain is:
    `psgc_code × population_year × source_version`
    
    There should be at most one row at this grain.

In [0]:
%sql

SELECT
    psgc_code,
    population_year,
    source_version,
    COUNT(*) AS row_count
FROM psgc_population_parsed
GROUP BY
    psgc_code,
    population_year,
    source_version
HAVING COUNT(*) > 1
ORDER BY
    row_count DESC,
    psgc_code,
    population_year;

5. Validate population values

    This check shows the range of parsed numeric values and the number of missing or negative values. Missing numeric values are expected when the parse status is MISSING or INVALID.

In [0]:
%sql

SELECT
    COUNT(*) AS total_observations,
    COUNT_IF(population_parse_status = 'VALID') AS valid_population,
    COUNT_IF(population_parse_status = 'MISSING') AS missing_population,
    COUNT_IF(population_parse_status = 'INVALID') AS invalid_population,
    COUNT_IF(population < 0) AS negative_population,
    COUNT_IF(population_parse_status = 'VALID_WITH_NOTE') AS valid_with_note,
    MIN(population) AS min_population,
    MAX(population) AS max_population
FROM psgc_population_parsed;

6. Validate accepted parse statuses

    The expected population parsing statuses are VALID, MISSING, and INVALID.

In [0]:
%sql

SELECT DISTINCT
    population_parse_status
FROM psgc_population_parsed
ORDER BY population_parse_status;

7. Create the PSGC population Silver table

The Silver table keeps all parsed population observations, including records with missing or invalid population values. The original value is retained in `population_raw`, and `population_parse_status` makes the data-quality state visible.

Lineage is preserved through `source_version` and `batch_id`.

In [0]:
%sql

CREATE OR REPLACE TABLE public_health.silver.population_silver
USING DELTA
COMMENT 'PSGC population Silver layer'
AS

SELECT
    psgc_code,
    population_year,
    population,
    population_raw,
    population_note,
    population_parse_status,
    source_version,
    content_checksum,
    batch_id,
    CURRENT_TIMESTAMP() AS silver_processed_at
FROM psgc_population_parsed;

8. Validate the created Silver table

In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT psgc_code) AS distinct_psgc_codes,
    COUNT(DISTINCT population_year) AS distinct_population_years,
    COUNT(DISTINCT source_version) AS distinct_source_versions,
    COUNT_IF(population_parse_status = 'VALID_WITH_NOTE') AS valid_with_note,
    COUNT_IF(population_parse_status = 'VALID') AS valid_population,
    COUNT_IF(population_parse_status = 'MISSING') AS missing_population,
    COUNT_IF(population_parse_status = 'INVALID') AS invalid_population
FROM public_health.silver.population_silver;

9. Check the final Silver schema

    Expected columns:

| Column | Purpose |
|---|---|
| `psgc_code` | PSGC geography key |
| `population_year` | Year of the population observation |
| `population` | Cleaned numeric population |
| `population_raw` | Original population value from PSGC |
| `population_parse_status` | `VALID`, `MISSING`, or `INVALID` |
| `source_version` | PSGC source version |
| `batch_id` | Bronze ingestion batch |
| `silver_processed_at` | Silver processing timestamp |

In [0]:
%sql

DESCRIBE public_health.silver.population_silver;

10. Sample the final Silver data

In [0]:
%sql

SELECT
    psgc_code,
    population_year,
    population,
    population_raw,
    population_parse_status,
    source_version,
    batch_id,
    silver_processed_at
FROM public_health.silver.population_silver
ORDER BY
    psgc_code,
    population_year
LIMIT 20;

In [0]:
%sql
WITH p AS (
  SELECT s.psgc_code, s.population_year, s.population, s.population_note, g.geographic_level
  FROM public_health.silver.population_silver s
  JOIN public_health.silver.geography_silver g USING (psgc_code)
)
SELECT pr.psgc_code, pr.population_year, pr.population AS province_total,
       l.lgu_sum, pr.population_note
FROM (SELECT * FROM p WHERE geographic_level = 'Prov') pr
JOIN (SELECT substr(psgc_code, 1, 5) AS prov, population_year, sum(population) AS lgu_sum
      FROM p WHERE geographic_level IN ('City', 'Mun') GROUP BY 1, 2) l
  ON l.prov = substr(pr.psgc_code, 1, 5) AND l.population_year = pr.population_year
WHERE NOT (pr.population <=> l.lgu_sum);